# Import Libraries


In [38]:
# %load_ext autoreload
%reload_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
from IPython import display

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold

In [39]:
# params_cfg = {
#     "action"  : "main_feat01",  
#     "seed"    : 42, # Set random seed
#     "exp_dir" : os.path.abspath('../../exps'),
#     'exp_name': 'featbase_03112025',
#     "data_dir": os.path.abspath("../../data/titanic"),
#     "verbose" : True,
# }

date = "29112025"

params_cfg = {
    "action"   : "train_feat01",  
    "feat_path": f"../../exps/featbase_{date}/data.npz",
    "seed"    : 42, # Set random seed
    "exp_dir" : os.path.abspath('../../exps'),
    'exp_name': f'train_{date}',
    "data_dir": os.path.abspath("../../../data"),
    "verbose" : True,
}
params_cfg.update(**{
    "save_dir": os.path.abspath(f'{params_cfg["exp_dir"]}/{params_cfg["exp_name"]}')
})

for v in params_cfg:
    print(f'+ {v}: {params_cfg[v]}')

globals().update(**params_cfg)

+ action: train_feat01
+ feat_path: ../../exps/featbase_29112025/data.npz
+ seed: 42
+ exp_dir: d:\dai_hoc\nam3\HK5\ML\LAB_GROUP\Challenge_3_Music_Genre\process\exps
+ exp_name: train_29112025
+ data_dir: d:\dai_hoc\nam3\HK5\ML\LAB_GROUP\Challenge_3_Music_Genre\data
+ verbose: True
+ save_dir: d:\dai_hoc\nam3\HK5\ML\LAB_GROUP\Challenge_3_Music_Genre\process\exps\train_29112025


# Data Load


In [40]:
df_train = pd.read_csv(f'{data_dir}/train.csv')
df_test = pd.read_csv(f'{data_dir}/test.csv')

if params_cfg["verbose"]:
    print("-"*10, "information", "-"*10)
    print(f'train-col: {set(df_train.columns)}')
    print(f'test-col: {set(df_test.columns)}')
    print("Union:", set(df_train.columns).intersection(set(df_test.columns)))
    print("Difference:", set(df_train.columns).difference(set(df_test.columns)))

---------- information ----------
train-col: {'valence', 'liveness', 'Class', 'key', 'time_signature', 'duration_in min/ms', 'acousticness', 'instrumentalness', 'Popularity', 'mode', 'energy', 'tempo', 'danceability', 'loudness', 'Id', 'speechiness', 'Artist Name', 'Track Name'}
test-col: {'valence', 'liveness', 'key', 'time_signature', 'duration_in min/ms', 'acousticness', 'instrumentalness', 'Popularity', 'mode', 'energy', 'tempo', 'danceability', 'loudness', 'Id', 'speechiness', 'Artist Name', 'Track Name'}
Union: {'valence', 'liveness', 'key', 'time_signature', 'duration_in min/ms', 'Track Name', 'instrumentalness', 'Popularity', 'mode', 'energy', 'tempo', 'danceability', 'loudness', 'Id', 'speechiness', 'Artist Name', 'acousticness'}
Difference: {'Class'}


# Preprocessing


## Check missing value


In [41]:
df_train.isna().sum()

Id                       0
Artist Name              0
Track Name               0
Popularity             333
danceability             0
energy                   0
key                   1609
loudness                 0
mode                     0
speechiness              0
acousticness             0
instrumentalness      3541
liveness                 0
valence                  0
tempo                    0
duration_in min/ms       0
time_signature           0
Class                    0
dtype: int64

In [42]:
df_test.isna().sum()

Id                      0
Artist Name             0
Track Name              0
Popularity             95
danceability            0
energy                  0
key                   405
loudness                0
mode                    0
speechiness             0
acousticness            0
instrumentalness      836
liveness                0
valence                 0
tempo                   0
duration_in min/ms      0
time_signature          0
dtype: int64

## Process missing value


In [43]:
def process_missing_value(df):
    df_output = df.copy()
    # Xử lý các giá trị missing value
    df_output['Popularity'] = df_output['Popularity'].fillna(df_output['Popularity'].median())
    df_output['key'] = df_output['key'].fillna(df_output['key'].median())
    
    def knn_imputer(df, col1, col2):
        from sklearn.impute import KNNImputer
        imputer = KNNImputer(n_neighbors=5)
        df_temp = df[[col1, col2]]
        df_temp[[col1, col2]] = imputer.fit_transform(df_temp[[col1, col2]])
        return df_temp[col1]
    
    df_output['instrumentalness'] = knn_imputer(df_output, 'instrumentalness', 'loudness')

    return df_output  


In [44]:
df_train.select_dtypes(include=['number']).columns.tolist()

['Id',
 'Popularity',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'duration_in min/ms',
 'time_signature',
 'Class']

In [45]:
final_feature = [
 'Popularity',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'duration_in min/ms',
 'time_signature',
 'Class'
]

# Feature Engineering


In [46]:
# ssss
def dummies_feature(df, feature_cols):
    df_output = df.copy()
    for col in feature_cols:
        dummies = pd.get_dummies(df_output[col], prefix=col, drop_first=True).astype(int)
        df_output = pd.concat([df_output, dummies], axis=1)
        df_output.drop(columns=[col], inplace=True)
    return df_output

In [47]:
def feature_engineering(df_data):
    df_output = df_data.copy()
    df_output['acoustic_energy'] = (df_output['acousticness'] * df_output['energy'])
    df_output['loudness_energy'] = (np.log1p(abs(df_output['loudness'])) * df_output['energy'])
    return df_output

In [48]:
def preprocessing_feature_01(df_data, is_train = True, is_debug = True, **kwargs):
    df_output = pd.DataFrame()
    df_output = df_data.copy()

    # Chuẩn bị các cột map và cut
    # for col in cls_feature.keys():
    #     df_output[col] = df_output[col].apply(lambda x: cls_feature[col].get(x, 0) if pd.notna(x) else cls_feature[col].get(df_output[col].mode()[0], 0))
    # for col in cut_feature.keys():
    #     df_output[col] = df_output[col].apply(cut_feature[col])

    # Xử lý missing value
    df_output = process_missing_value(df_output)
    # Drop các cột không sử dụng
    for x in df_train.columns:
        if x not in final_feature:
            df_output = df_output.drop(columns=[x], axis=1)

    # Feature engineering
    df_output = feature_engineering(df_output)

    # One-hot encoding cho các biến phân loại (nếu có)
    # df_output = dummies_feature(df_output, feature_cols=["key", "mode", "time_signature"])

    if is_train:
        df_output["Output"] = df_data["Class"]
        df_output = df_output.drop(columns=['Class'], axis=1)


    if is_debug:
        print("head(5)")
        display.display(df_output.head(5))
        print("isna")
        display.display(df_output.isna().sum())
        print("Columns:", df_output.columns.tolist())
        print("Shape:", df_output.shape)

    return df_output, None
    pass

# df_train = pd.read_csv(f'{data_dir}/train.csv')
# preprocessing_feature_01(df_train)

# Main


In [49]:
def main_feat01(**kwargs):
    # load data
    df_train = pd.read_csv(f'{data_dir}/train.csv')
    df_test = pd.read_csv(f'{data_dir}/test.csv')
    # preprocessing
    df_output_train, _ = preprocessing_feature_01(df_train, is_train=True, is_debug=False)
    df_output_test, _ = preprocessing_feature_01(df_test, is_train=False, is_debug=False)
    
    # saving
    os.makedirs(save_dir, exist_ok=True)
    
    # Lưu trực tiếp DataFrame objects (cần allow_pickle=True khi load)
    np.savez(f'{save_dir}/data.npz', 
             train_data=df_output_train.values,
             test_data=df_output_test.values,
             train_columns=df_output_train.columns.values,  # Lưu tên cột train
             test_columns=df_output_test.columns.values,
             allow_pickle=True)
    
    print("Đã lưu DataFrame với đầy đủ thông tin cột")

    kwargs.get('global_cfg', {}).update(**locals())

if params_cfg["action"] == "train_feat01":
    print("Runing ... [train_feat01]")
    main_feat01(global_cfg = globals())

Runing ... [train_feat01]
Đã lưu DataFrame với đầy đủ thông tin cột


# End
